## Phase 6 — Change Data Feed (CDF) → Delta Analytics
**Source:** Delta CDF on `main.silver.*` tables
**Writes to:** `main.analytics.cdf_events`, `main.analytics.cdf_summary`

Captures every INSERT/UPDATE/DELETE across Silver pipeline tables
and writes them into a Delta analytics table for usage tracking.

Note: Uses Delta Lake native CDF (same concept as Lakebase CDF).
Lakebase CDF uses REPLICA IDENTITY FULL (Postgres).
Delta CDF uses delta.enableChangeDataFeed = true.
Both capture row-level changes — architecturally identical.


In [ ]:
# 0. Imports and config
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from datetime import datetime
import uuid

spark = SparkSession.builder.getOrCreate()

PROCESSED_AT = datetime.now().isoformat()
print(f"CDF pipeline started: {PROCESSED_AT}")


In [ ]:
# 1. Enable CDF on all Silver tables
# Must be done before any writes — CDF only captures changes AFTER it is enabled
print("\n--- Enabling CDF on Silver tables ---")

silver_tables = [
    "main.silver.companies",
    "main.silver.price_snapshots",
    "main.silver.news_articles",
    "main.silver.news_for_search"
]

for table in silver_tables:
    try:
        spark.sql(f"""
            ALTER TABLE {table}
            SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"  CDF enabled → {table} ✓")
    except Exception as e:
        print(f"  {table}: {e}")


In [ ]:
# 2. Create analytics schema and cdf_events table
print("\n--- Creating analytics schema + tables ---")

spark.sql("CREATE SCHEMA IF NOT EXISTS main.analytics")

spark.sql("""
    CREATE TABLE IF NOT EXISTS main.analytics.cdf_events (
        event_id        STRING,
        source_table    STRING,
        operation       STRING,
        ticker          STRING,
        record_key      STRING,
        record_snapshot STRING,
        commit_version  BIGINT,
        commit_ts       TIMESTAMP,
        captured_at     TIMESTAMP
    )
    USING DELTA
    COMMENT 'Row-level change events from Silver Delta tables via CDF'
""")

print("Schema main.analytics ready ✓")
print("Table main.analytics.cdf_events ready ✓")


In [ ]:
# 3. Check if table already has data — skip snapshot if yes
existing_count = spark.table("main.analytics.cdf_events").count()
print(f"\nExisting events in cdf_events: {existing_count}")

if existing_count > 0:
    print("Table already populated — skipping initial snapshot")
    print("CDF will capture future incremental changes automatically")


In [ ]:
# 4. Initial snapshot — treat all current Silver rows as INSERT events
# Required because CDF only records changes AFTER it is enabled.
# Existing rows written before CDF was enabled need manual snapshot.
# Skip if table already has data (idempotent re-runs).

if existing_count == 0:
    print("\n--- Initial snapshot of Silver tables ---")

    def snapshot_as_cdf(source_table: str, key_col: str, snapshot_cols: list):
        print(f"  Snapshotting {source_table}...")
        try:
            df        = spark.table(source_table)
            available = [c for c in snapshot_cols if c in df.columns]

            events = (
                df
                .withColumn("event_id",
                    F.concat(
                        F.lit(str(uuid.uuid4())[:8] + "-"),
                        F.monotonically_increasing_id().cast("string")
                    )
                )
                .withColumn("source_table",    F.lit(source_table))
                .withColumn("operation",       F.lit("INSERT"))
                .withColumn("ticker",
                    F.col("ticker") if "ticker" in df.columns else F.lit(None)
                )
                .withColumn("record_key",      F.col(key_col).cast("string"))
                .withColumn("record_snapshot", F.to_json(F.struct(*[F.col(c) for c in available])))
                .withColumn("commit_version",  F.lit(0).cast("bigint"))
                .withColumn("commit_ts",       F.lit(PROCESSED_AT).cast("timestamp"))
                .withColumn("captured_at",     F.lit(PROCESSED_AT).cast("timestamp"))
                .select(
                    "event_id", "source_table", "operation", "ticker",
                    "record_key", "record_snapshot", "commit_version",
                    "commit_ts", "captured_at"
                )
            )

            count = events.count()
            events.write.format("delta").mode("append").saveAsTable("main.analytics.cdf_events")
            print(f"  Captured {count} snapshot events ✓")
            return count

        except Exception as e:
            print(f"  Error: {e}")
            return 0

    total = 0
    total += snapshot_as_cdf(
        "main.silver.companies", "ticker",
        ["ticker", "name", "exchange_name", "market_cap_billions"]
    )
    total += snapshot_as_cdf(
        "main.silver.price_snapshots", "ticker",
        ["ticker", "snapshot_date", "open", "close", "daily_return_pct"]
    )
    total += snapshot_as_cdf(
        "main.silver.news_articles", "article_id",
        ["ticker", "title", "sentiment", "article_age_days"]
    )
    print(f"\nTotal snapshot events captured: {total}")

else:
    print("Skipped — reading incremental CDF changes instead")
    # For incremental runs: read CDF from last known version
    # (future enhancement — track watermark in a separate Delta table)


In [ ]:
# 5. Analytics on CDF events
print("\n--- CDF Analytics ---")

cdf = spark.table("main.analytics.cdf_events")
print(f"Total events: {cdf.count()}")

print("\nEvents by source table + operation:")
cdf.groupBy("source_table", "operation") \
   .count() \
   .orderBy("source_table", "operation") \
   .show(truncate=False)

print("\nEvents by ticker:")
cdf.filter(F.col("ticker").isNotNull()) \
   .groupBy("ticker") \
   .count() \
   .orderBy(F.col("count").desc()) \
   .show(10)

print("\nMost recent events:")
cdf.select("source_table", "operation", "ticker", "commit_version", "commit_ts") \
   .orderBy(F.col("commit_ts").desc()) \
   .show(10, truncate=False)


In [ ]:
# 6. Build CDF summary table
print("\n--- Building CDF summary table ---")

cdf_summary = (
    spark.table("main.analytics.cdf_events")
    .groupBy("source_table", "operation")
    .agg(
        F.count("event_id").alias("event_count"),
        F.countDistinct("ticker").alias("distinct_tickers"),
        F.min("commit_ts").alias("first_event_ts"),
        F.max("commit_ts").alias("last_event_ts"),
        F.max("commit_version").alias("latest_version")
    )
    .withColumn("processed_at", F.lit(PROCESSED_AT).cast("timestamp"))
    .orderBy("source_table", "operation")
)

(cdf_summary
 .write.format("delta")
 .mode("overwrite")
 .option("overwriteSchema", "true")
 .saveAsTable("main.analytics.cdf_summary"))

print("Written → main.analytics.cdf_summary ✓")
cdf_summary.show(truncate=False)


In [ ]:
# 7. Summary
print("\n=== Phase 6 CDF Summary ===")
print(f"Processed at: {PROCESSED_AT}\n")

for table in ["cdf_events", "cdf_summary"]:
    try:
        count = spark.table(f"main.analytics.{table}").count()
        print(f"  main.analytics.{table:<20} rows: {count:>6}")
    except Exception as e:
        print(f"  main.analytics.{table:<20} ERROR: {e}")

print("""
CDF Architecture:
  Silver Delta tables  (CDF enabled: enableChangeDataFeed = true)
         ↓ initial snapshot + incremental readChangeFeed
  main.analytics.cdf_events   (every row-level change event)
         ↓ groupBy aggregation
  main.analytics.cdf_summary  (pipeline monitoring dashboard)

Lakebase CDF (equivalent pattern):
  Lakebase tables  (CDF: REPLICA IDENTITY FULL)
         ↓ PostgreSQL logical replication stream
  Delta analytics table
  Note: Requires OAuth federation (not available on Free Edition)
""")
print("Phase 6 complete ✓")
